# pandas.DataFrame -- завантаження даних з html та попередня обробка

In [ ]:
import requests
from bs4 import BeautifulSoup
import html5lib
import pandas as pd
import numpy as np
from pathlib import Path
import io
pd.__version__

'3.0.0'

## Приклад 1. Щільність населення

Використовуються дані, завантажені в <code>bs-1.ipynb</code>.

Ініціюємо html-дерево.

In [ ]:
# url = 'https://en.wikipedia.org/wiki/World_population'
fname = Path('prepare', 'density.html')
with open(fname,  encoding='utf-8') as f:
    data = f.read()
soup = BeautifulSoup(data, 'html.parser')

Серед таблиць шукаємо ту, що містить інформацію про щільність населення. Погук базується на фразі з заголовку таблиці.

In [ ]:
tables = soup.find_all('table')
tables[0]
for index,table in enumerate(tables):
    if ("10 most densely populated countries" in str(table)):
        table_index = index
        break

Ініціюємо текстовий потік, асоційований з рядком, що містить вміст таблиці. В якості парсера використовуємо зв'язку beautifulsoap+html5lib.
Оскільки на вхід дається рівно одна таблиця, то результуючий список містить рівно один елемент. Його і використовуємо.

In [ ]:
density = pd.read_html(io.StringIO(str(tables[table_index])), flavor='bs4')[0]
density

,Rank,Country,Population,Area (km2),Density (pop/km2)
0,1,Singapore,5921231,719,8235
1,2,Bangladesh,165650475,148460,1116
2,3,Palestine[note 3][103],5223000,6025,867
3,4,Taiwan[note 4],23580712,35980,655
4,5,South Korea,51844834,99720,520
5,6,Lebanon,5296814,10400,509
6,7,Rwanda,13173730,26338,500
7,8,Burundi,12696478,27830,456
8,9,Israel,9402617,21937,429
9,10,India,1389637446,3287263,423


In [ ]:
density.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Rank               10 non-null     int64
 1   Country            10 non-null     str  
 2   Population         10 non-null     int64
 3   Area (km2)         10 non-null     int64
 4   Density (pop/km2)  10 non-null     int64
dtypes: int64(4), str(1)
memory usage: 532.0 bytes


In [ ]:
density.set_index('Rank', inplace=True)

In [ ]:
density.tail()

,Country,Population,Area (km2),Density (pop/km2)
Rank,,,,
6,Lebanon,5296814,10400,509
7,Rwanda,13173730,26338,500
8,Burundi,12696478,27830,456
9,Israel,9402617,21937,429
10,India,1389637446,3287263,423


In [ ]:
density['calc_density'] = density.Population/density['Area (km2)']
density

,Country,Population,Area (km2),Density (pop/km2),calc_density
Rank,,,,,
1,Singapore,5921231,719,8235,8235.369958
2,Bangladesh,165650475,148460,1116,1115.791964
3,Palestine[note 3][103],5223000,6025,867,866.887967
4,Taiwan[note 4],23580712,35980,655,655.383880
5,South Korea,51844834,99720,520,519.904071
6,Lebanon,5296814,10400,509,509.309038
7,Rwanda,13173730,26338,500,500.179588
8,Burundi,12696478,27830,456,456.215523
9,Israel,9402617,21937,429,428.619091


Перевіримо, чи наявні помилки обрахунку щільності у таблиці.

In [ ]:
((density['calc_density']-density['Density (pop/km2)']).abs().round(0)>1).any()

np.False_

Жодне зі значень не перевищує 1. Це свідчить, що розбіжність даних таблиці з обчисленими на рівні похибки округлення.

## Приклад 2. Площа країн

In [ ]:
# url = 'https://en.wikipedia.org/wiki/List_of_European_countries_by_area'
fname = Path('prepare', 'area.html')
with open(fname,  encoding='utf-8') as f:
    data = f.read()
soup = BeautifulSoup(data, 'html.parser')

In [ ]:
tables = soup.find_all('table')
area = pd.read_html(io.StringIO(str(tables[0])), flavor='bs4')[0]
area

Unnamed: 0_level_0       Country or dependency % total Europe area  \
   Unnamed: 0_level_1       Country or dependency % total         km2   
0                   –                      Europe    100%  10014000.0   
1                 1 K                      Russia   39.5%   3952550.0   
2                   2                     Ukraine    6.0%    603549.0   
3                 3 T                      France    5.4%    543941.0   
4                 4 T                       Spain    5.0%    498485.0   
..                ...                         ...     ...         ...   
58               49 C                     Armenia      0%     29743.0   
59               50 C                      Cyprus      0%      9251.0   
60                – C                    Abkhazia      0%      8665.0   
61                – C               South Ossetia      0%      3885.0   
62                – C  Akrotiri and Dhekelia (UK)      0%       254.0   

              Unnamed: 5_level_0  
          mi2 Unnamed: 5_level_1  
0   3866000.0                NaN  
1   1526090.0                [a]  
2    233032.0                [b]  
3    210017.0                [c]  
4    192466.0                [d]  
..        ...                ...  
58    11484.0                [z]  
59     3572.0               [aa]  
60     3346.0               [ab]  
61     1500.0               [ac]  
62       98.0               [ad]  

[63 rows x 6 columns]

In [ ]:
area.info()

<class 'pandas.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 6 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   (Unnamed: 0_level_0, Unnamed: 0_level_1)        63 non-null     str    
 1   (Country or dependency, Country or dependency)  63 non-null     str    
 2   (% total, % total)                              62 non-null     str    
 3   (Europe area, km2)                              63 non-null     float64
 4   (Europe area, mi2)                              63 non-null     float64
 5   (Unnamed: 5_level_0, Unnamed: 5_level_1)        30 non-null     str    
dtypes: float64(2), str(4)
memory usage: 3.1 KB


In [ ]:
area.columns

MultiIndex([(   'Unnamed: 0_level_0',    'Unnamed: 0_level_1'),
            ('Country or dependency', 'Country or dependency'),
            (              '% total',               '% total'),
            (          'Europe area',                   'km2'),
            (          'Europe area',                   'mi2'),
            (   'Unnamed: 5_level_0',    'Unnamed: 5_level_1')],
           )

In [ ]:
area.loc[2, ('Europe area', 'km2')]

np.float64(603549.0)

In [ ]:
area.columns.levels

FrozenList([['% total', 'Country or dependency', 'Europe area', 'Unnamed: 0_level_0', 'Unnamed: 5_level_0'], ['% total', 'Country or dependency', 'Unnamed: 0_level_1', 'Unnamed: 5_level_1', 'km2', 'mi2']])

In [ ]:
area.columns.droplevel()

Index(['Unnamed: 0_level_1', 'Country or dependency', '% total', 'km2', 'mi2',
       'Unnamed: 5_level_1'],
      dtype='str')

In [ ]:
area.columns.droplevel(1)

Index(['Unnamed: 0_level_0', 'Country or dependency', '% total', 'Europe area',
       'Europe area', 'Unnamed: 5_level_0'],
      dtype='str')

In [ ]:
area.columns = area.columns.to_flat_index()
area.columns

Index([      ('Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
       ('Country or dependency', 'Country or dependency'),
                                   ('% total', '% total'),
                                   ('Europe area', 'km2'),
                                   ('Europe area', 'mi2'),
             ('Unnamed: 5_level_0', 'Unnamed: 5_level_1')],
      dtype='object')

In [ ]:
area.drop(columns=('Europe area', 'mi2'), inplace=True)
area.head()

,"(Unnamed: 0_level_0, Unnamed: 0_level_1)","(Country or dependency, Country or dependency)","(% total, % total)","(Europe area, km2)","(Unnamed: 5_level_0, Unnamed: 5_level_1)"
0,–,Europe,100%,10014000.0,NaN
1,1 K,Russia,39.5%,3952550.0,[a]
2,2,Ukraine,6.0%,603549.0,[b]
3,3 T,France,5.4%,543941.0,[c]
4,4 T,Spain,5.0%,498485.0,[d]


In [ ]:
area.columns = ['Comment', 'State', 'Total%',  'km2', 'Notes']
area.tail()

,Comment,State,Total%,km2,Notes
58,49 C,Armenia,0%,29743.0,[z]
59,50 C,Cyprus,0%,9251.0,[aa]
60,– C,Abkhazia,0%,8665.0,[ab]
61,– C,South Ossetia,0%,3885.0,[ac]
62,– C,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.drop(columns='Comment', inplace=True)
area.head()

,State,Total%,km2,Notes
0,Europe,100%,10014000.0,NaN
1,Russia,39.5%,3952550.0,[a]
2,Ukraine,6.0%,603549.0,[b]
3,France,5.4%,543941.0,[c]
4,Spain,5.0%,498485.0,[d]


In [ ]:
area.replace('Total%', np.nan, inplace=True)
area.tail()

,State,Total%,km2,Notes
58,Armenia,0%,29743.0,[z]
59,Cyprus,0%,9251.0,[aa]
60,Abkhazia,0%,8665.0,[ab]
61,South Ossetia,0%,3885.0,[ac]
62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.reset_index(inplace=True)
area.head()

,index,State,Total%,km2,Notes
0,0,Europe,100%,10014000.0,NaN
1,1,Russia,39.5%,3952550.0,[a]
2,2,Ukraine,6.0%,603549.0,[b]
3,3,France,5.4%,543941.0,[c]
4,4,Spain,5.0%,498485.0,[d]


In [ ]:
lst = list(area.columns)
lst[0] = 'Rank'
area.columns = lst
area.head()

,Rank,State,Total%,km2,Notes
0,0,Europe,100%,10014000.0,NaN
1,1,Russia,39.5%,3952550.0,[a]
2,2,Ukraine,6.0%,603549.0,[b]
3,3,France,5.4%,543941.0,[c]
4,4,Spain,5.0%,498485.0,[d]


In [ ]:
area['Rank'] = area['Rank'].astype('Int64')
area.tail()

,Rank,State,Total%,km2,Notes
58,58,Armenia,0%,29743.0,[z]
59,59,Cyprus,0%,9251.0,[aa]
60,60,Abkhazia,0%,8665.0,[ab]
61,61,South Ossetia,0%,3885.0,[ac]
62,62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.dropna(subset=['Rank', 'State'], inplace=True)
area.tail()

,Rank,State,Total%,km2,Notes
58,58,Armenia,0%,29743.0,[z]
59,59,Cyprus,0%,9251.0,[aa]
60,60,Abkhazia,0%,8665.0,[ab]
61,61,South Ossetia,0%,3885.0,[ac]
62,62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.km2 = area.km2.replace(0.0, np.nan)
area.tail()

,Rank,State,Total%,km2,Notes
58,58,Armenia,0%,29743.0,[z]
59,59,Cyprus,0%,9251.0,[aa]
60,60,Abkhazia,0%,8665.0,[ab]
61,61,South Ossetia,0%,3885.0,[ac]
62,62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.dropna(subset='Total%', inplace=True)
area.tail()

,Rank,State,Total%,km2,Notes
58,58,Armenia,0%,29743.0,[z]
59,59,Cyprus,0%,9251.0,[aa]
60,60,Abkhazia,0%,8665.0,[ab]
61,61,South Ossetia,0%,3885.0,[ac]
62,62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


In [ ]:
area.State = area.State.str.replace('*', '')
area.tail()

,Rank,State,Total%,km2,Notes
58,58,Armenia,0%,29743.0,[z]
59,59,Cyprus,0%,9251.0,[aa]
60,60,Abkhazia,0%,8665.0,[ab]
61,61,South Ossetia,0%,3885.0,[ac]
62,62,Akrotiri and Dhekelia (UK),0%,254.0,[ad]


З моменту першого оброблення цієї сторінки до поточного пройшов хоч невеликий, але час, за який таблиця дещо змінилася.